选取了TSPLIB数据集中的gr21.tsp \
TSPLIB95是由海德堡大学提供的旅行商问题（TSP）标准测试数据集，包含多种城市规模、分布类型的对称和非对称TSP实例，以及相关最优解或已知最优解，广泛用于算法性能评估和优化研究。\
其中gr21.tsp提供了21城市对称tsp的数据。对称tsp，即城市 i 到 j 的距离 = 城市 j 到 i 的距离。该问题官方给出了最优路径距离，即2707。

In [1]:
import tsplib95
import numpy as np 
import sys
import time
import random

读取数据

In [2]:
BEST_DISTANCE = 2707                      # 官方给出的最优路径距离
problem = tsplib95.load('gr21.tsp')       # 读取tsp文件
node_count = problem.dimension            # 节点数量

matrix = np.full((node_count, node_count), np.inf)  # 初始化距离为无穷大
for i in range(node_count):
    for j in range(node_count):
        matrix[i][j] = problem.get_weight(i, j)
print(f"邻接矩阵: \n{matrix}")

邻接矩阵: 
[[  0. 510. 635.  91. 385. 155. 110. 130. 490. 370. 155.  68. 610. 655.
  480. 265. 255. 450. 170. 240. 380.]
 [510.   0. 355. 415. 585. 475. 480. 500. 605. 320. 380. 440. 360. 235.
   81. 480. 440. 270. 445. 290. 140.]
 [635. 355.   0. 605. 390. 495. 570. 540. 295. 700. 640. 575. 705. 585.
  435. 420. 755. 625. 750. 590. 495.]
 [ 91. 415. 605.   0. 350. 120.  78.  97. 460. 280.  63.  27. 520. 555.
  380. 235. 235. 345. 160. 140. 280.]
 [385. 585. 390. 350.   0. 240. 320. 285. 120. 590. 430. 320. 835. 750.
  575. 125. 650. 660. 495. 480. 480.]
 [155. 475. 495. 120. 240.   0.  96.  36. 350. 365. 200.  91. 605. 615.
  440. 125. 370. 430. 265. 255. 340.]
 [110. 480. 570.  78. 320.  96.   0.  29. 425. 350. 160.  48. 590. 625.
  455. 200. 320. 420. 220. 205. 350.]
 [130. 500. 540.  97. 285.  36.  29.   0. 390. 370. 175.  67. 610. 645.
  465. 165. 350. 440. 240. 220. 370.]
 [490. 605. 295. 460. 120. 350. 425. 390.   0. 625. 535. 430. 865. 775.
  600. 230. 680. 690. 600. 515. 505.]
 [3

先使用dfs求解tsp问题 \
超过5min未能求得最优解(实际上所需时间是天文级数字)

In [3]:
# dfs函数中调用了很多外部全局变量
min_distance = sys.maxsize      # 记录最短路径的距离和路径
best_path = []
visited = [False] * node_count  # 记录城市是否访问，True已访问

start_time = time.time()        # 判断dfs求解tsp问题是否超时
timeout = 5 * 60                # 超时时间，5min

def dfs(current, count, distance, path):
    """ 
    深度优先遍历求解旅行商问题

    :param current: 当前节点索引
    :param count: 已访问城市数, 包括当前节点
    :param distance: 路径距离
    :param path: 旅行商具体路径
    """
    # 更新全局最优值，找到最优解
    global min_distance, best_path     # 用global能在函数内修改全局变量的值，否则只能读取全局变量(针对不可变类型生效)
    if time.time() - start_time > timeout:
        raise TimeoutError(f"dfs求解tsp问题已超时,(超过{int(timeout/60)}min)")
    # 如果已访问所有城市，且与起始城市相连, 即求解出一条TSP问题的可行解
    # 起始城市可选0节点
    if count == node_count and matrix[current][0] != float('inf'):
        if distance + matrix[current][0] < min_distance:
            min_distance =  distance + matrix[current][0] 
            best_path = path + [0]
        return 
    # 深度优先遍历
    # 遍历所有未访问且当前节点能到达的城市
    for i in range(node_count):
        if not visited[i] and matrix[current][i] != float('inf'):
            # python中可变类型如列表、字典等，修改变量内部的元素值，可以直接在函数内操作，不需要global
            visited[i] = True           # 访问节点
            dfs(i, count+1, distance+matrix[current][i], path+[i]) 
            visited[i] = False          # 回溯到未访问该节点时



visited[0] = True     # 起始城市选0节点
print(f"官方给出的最短路径距离: {BEST_DISTANCE}")
try:
    dfs(0, 1, 0, [0])
    print(f"最短距离: {min_distance}")
    print(f"最短路径: {best_path}")
except TimeoutError as e:
    print(e)
    print(f"超时前找到的最短距离: {min_distance if min_distance != sys.maxsize else '未找到可行解'}")
    print(f"超时前找到的最短路径: {best_path if best_path else '未找到可行解'}")

官方给出的最短路径距离: 2707
dfs求解tsp问题已超时,(超过5min)
超时前找到的最短距离: 5207.0
超时前找到的最短路径: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 12, 13, 14, 20, 17, 16, 18, 19, 10, 11, 15, 0]


再采取模拟退火算法，试图求得tsp问题的近似最优解

In [4]:
# 需要的函数      
def cal_distance(matrix, path):
    """
    计算TSP可行路径的总距离
    
    :param matrix: 城市邻接矩阵
    :param path: 可行路径
    :return distance: 路径总距离
    """
    distance = 0
    for i in range(len(matrix)):
        distance += matrix[path[i]][path[i+1]]
    return distance

def change_two_cities(path):
    """  
    随机交换两个城市位置生成新路径

    :param path: 可行路径
    :return new_path: 新可行路径
    """
    new_path = path.copy()                           # 复制，不修改原路径
    i, j = random.sample(range(1, len(path)-1), 2)   # 从列表中随机抽取不重复元素, path头尾固定为城市0，不用交换
    new_path[i], new_path[j] = new_path[j], new_path[i]
    return new_path

def reverse_subpath(path):
    """
    将路径切分为两条子路径, 其中一条翻转后再次拼接, 生成新路径
    
    :param path: 可行路径
    :return new_path: 新可行路径
    """
    i, j = random.sample(range(1, len(path)-1), 2)   # [1, 20] 随机抽取不重复元素
    if i > j:         
        i, j = j, i  # 同时交换

    new_path = path.copy()
    subpath = new_path[i:j+1]
    new_path = new_path[:i] + subpath[::-1] + new_path[j+1:]     
    return new_path

In [12]:
T0 = 100        # 初始温度
t = T0          # 每轮的温度
T_end = 0.01    # 最终温度
K = 0.99        # 温度衰减系数

# L 每轮温度，生成新可行解的数量, 设定为温度越高L越大(最大300)，温度最低L越小(最小50)
# 高温多探索扩大搜索区域，低温少探索节约时间
L_max = 300
L_min = 50
L = int((t/T0) * (L_max-L_min) + L_min)   
# 由邻接矩阵可以看出，每个城市间都可以通行，只是距离不同
# 可以直接选0，1，2，..., 20，0作为初始路径(一个可行解)
# 也可以选dfs已求解出来的一个可行解
# 初始化最优路径、距离
best_path = [i for i in range(21)] + [0]  
min_distance = cal_distance(matrix, best_path)
path = best_path           # 当前路径
distance = min_distance    # 当前距离
print(f"初始化路径: {best_path}")
print(f"初始化路径的距离: {min_distance}")

初始化路径: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 0]
初始化路径的距离: 6620.0


In [13]:
# 模拟退火算法求解tsp问题
start_time = time.time()
epoch = 0
while t > T_end:            # 每轮温度不断下降
    for _ in range(L):      # L 每轮温度，生成新可行解的数量
        # 根据当前路径，用两种方法构造新路径
        if random.random() < 0.5:            # random.random() 生成一个01之间的随机小数
            new_path = change_two_cities(path)
        else:
            new_path = reverse_subpath(path)
        new_distance = cal_distance(matrix, new_path)
        delta = new_distance - distance      # 新可行解和当前可行解之间的路径距离之差
        if delta < 0:                        # 当新解更优时，直接把新解作为当前解
            path = new_path
            distance = new_distance
            if distance < min_distance:      # 更新全局最优解
                best_path = path 
                min_distance = distance
        else:                                # 当新解更差时，仍有一定概率将新解作为当前解
            if random.random() < np.exp(-delta/t):    # 新解越差，接受概率越小；温度越高，接受概率越大
                path = new_path
                distance = new_distance

    t *= K                                   # 降温
    L = int((t/T0) * (L_max-L_min) + L_min)  # 更新L  
    epoch += 1
    print(f"第{epoch}轮, 温度为{t:.2f}, 最短距离: {min_distance}")
    print(f"第{epoch}轮, 温度为{t:.2f}, 最短路径: {best_path}")
    if min_distance == 2707:
        break

end_time = time.time()
print(f"官方给出的最短路径距离: {BEST_DISTANCE}")
print(f"运行时间: {(end_time-start_time)/60:.2f}min")
print(f"最短距离: {min_distance}")
print(f"最短路径: {best_path}")

第1轮, 温度为99.00, 最短距离: 3557.0
第1轮, 温度为99.00, 最短路径: [0, 18, 16, 9, 19, 20, 14, 17, 12, 13, 1, 2, 8, 15, 6, 7, 5, 4, 11, 3, 10, 0]
第2轮, 温度为98.01, 最短距离: 3557.0
第2轮, 温度为98.01, 最短路径: [0, 18, 16, 9, 19, 20, 14, 17, 12, 13, 1, 2, 8, 15, 6, 7, 5, 4, 11, 3, 10, 0]
第3轮, 温度为97.03, 最短距离: 3472.0
第3轮, 温度为97.03, 最短路径: [0, 10, 3, 5, 15, 4, 8, 2, 1, 12, 13, 9, 17, 14, 20, 19, 16, 18, 6, 11, 7, 0]
第4轮, 温度为96.06, 最短距离: 3428.0
第4轮, 温度为96.06, 最短路径: [0, 11, 10, 17, 13, 12, 9, 19, 14, 20, 1, 2, 8, 4, 15, 5, 7, 6, 3, 16, 18, 0]
第5轮, 温度为95.10, 最短距离: 3428.0
第5轮, 温度为95.10, 最短路径: [0, 11, 10, 17, 13, 12, 9, 19, 14, 20, 1, 2, 8, 4, 15, 5, 7, 6, 3, 16, 18, 0]
第6轮, 温度为94.15, 最短距离: 3113.0
第6轮, 温度为94.15, 最短路径: [0, 6, 7, 11, 5, 15, 4, 8, 2, 1, 14, 20, 9, 17, 13, 12, 16, 18, 10, 19, 3, 0]
第7轮, 温度为93.21, 最短距离: 3113.0
第7轮, 温度为93.21, 最短路径: [0, 6, 7, 11, 5, 15, 4, 8, 2, 1, 14, 20, 9, 17, 13, 12, 16, 18, 10, 19, 3, 0]
第8轮, 温度为92.27, 最短距离: 3113.0
第8轮, 温度为92.27, 最短路径: [0, 6, 7, 11, 5, 15, 4, 8, 2, 1, 14, 20, 9, 17, 13, 12, 16, 18